# 02 - Prompts and Output Parsers

Covers: prompt templates, list / JSON / Pydantic output parsers, and `with_structured_output`.

In [ ]:
from llm_engineering import get_llm

llm = get_llm()

## Prompt templates

In [ ]:
from langchain_core.prompts import PromptTemplate

email_template = PromptTemplate.from_template(
    "Create an invitation email to the recipient that is {recipient_name} "
    "for an event that is {event_type} "
    "in a language that is {language}. "
    "Mention the event location that is {event_location} "
    "and event date that is {event_date}. "
    "Also write a few sentences about the event description that is {event_description} "
    "in a style that is {style}."
)

details = {
    "recipient_name": "John",
    "event_type": "product launch",
    "language": "American English",
    "event_location": "Grand Ballroom, City Center Hotel",
    "event_date": "11 AM, January 15, 2024",
    "event_description": "an exciting unveiling of our latest GenAI product",
    "style": "enthusiastic tone",
}

prompt_value = email_template.invoke(details)
response = llm.invoke(prompt_value)
response.content

## Output parsers

Each parser provides `format_instructions` that tell the model which format to answer in, and a `parse()` method that converts the raw text into a Python object.

### Comma-separated list

In [ ]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

list_parser = CommaSeparatedListOutputParser()

list_prompt = PromptTemplate.from_template(
    template="Answer the question.\n{format_instructions}\n{question}",
    partial_variables={"format_instructions": list_parser.get_format_instructions()},
)

prompt_value = list_prompt.invoke({"question": "List 4 chocolate brands"})
response = llm.invoke(prompt_value)
list_parser.parse(response.content)

### JSON

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()

json_prompt = PromptTemplate.from_template(
    template="Answer the question.\n{format_instructions}\n{question}",
    partial_variables={"format_instructions": json_parser.get_format_instructions()},
)

prompt_value = json_prompt.invoke(
    {"question": "Give the title, director, and release year of the movie Inception."}
)
response = llm.invoke(prompt_value)
json_parser.parse(response.content)

### Pydantic

Validates the output against a schema and returns a typed object.

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field


class Author(BaseModel):
    name: str = Field(description="The name of the author")
    number: int = Field(description="The number of books written by the author")
    books: list[str] = Field(description="The list of books they wrote")


pydantic_parser = PydanticOutputParser(pydantic_object=Author)

author_prompt = PromptTemplate.from_template(
    template="Answer the question.\n{format_instructions}\n{question}",
    partial_variables={"format_instructions": pydantic_parser.get_format_instructions()},
)

prompt_value = author_prompt.invoke({"question": "Generate the books written by Dan Brown."})
response = llm.invoke(prompt_value)
pydantic_parser.parse(response.content)

## `with_structured_output`

Skips manual format instructions: the model is asked for the schema directly (using native structured output / tool calling when the provider supports it) and returns the parsed object.

In [ ]:
structured_llm = llm.with_structured_output(Author)
structured_llm.invoke("Generate the books written by Dan Brown.")

In [ ]:
from datetime import datetime


class DateAnswer(BaseModel):
    value: datetime = Field(description="The date and time being asked about, in ISO 8601 format")


date_llm = llm.with_structured_output(DateAnswer)
result = date_llm.invoke("When was the iPhone released?")
print(result.value, type(result.value))